# Exp2 — SDXL GenEval Runner (Colab)

Compares **Standard Sampling**, **Resampling (RePaint / Lugmayr 2022)**, and **Z-Sampling** on the GenEval benchmark using SDXL.

Outputs are saved directly to Google Drive so they survive session disconnections.  
Re-running any cell is safe — already-generated images are skipped automatically.

**Before running:** Go to `Runtime → Change runtime type → GPU` (T4 is free, A100 requires Colab Pro).

## Cell 1 — Mount Google Drive and clone repo

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import subprocess, os

REPO_DIR = "/content/generative-modelling"
BRANCH = "exp2-resampling-feature"

if not os.path.exists(REPO_DIR):
    subprocess.run(
        ["git", "clone", "https://github.com/viftode4/generative-modelling.git", REPO_DIR],
        check=True
    )

subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin"], check=True)
subprocess.run(["git", "-C", REPO_DIR, "checkout", BRANCH], check=True)
subprocess.run(["git", "-C", REPO_DIR, "pull", "origin", BRANCH], check=True)

print(f"Repo ready at {REPO_DIR} on branch {BRANCH}")

## Cell 2 — Install dependencies

Uses diffusers 0.36+ (not the pinned 0.22.3 from `requirements-zigzag.txt`, which is for the upstream ZigZag code and is too old for our resampling rewrite).

In [ ]:
!pip install -q diffusers transformers accelerate safetensors tqdm pandas xformers
print("Dependencies installed.")

## Cell 3 — Smoke test (2 prompts)

Run this first to confirm nothing crashes before committing to the full run.

In [ ]:
import sys, os
os.chdir(REPO_DIR)
sys.path.insert(0, os.path.join(REPO_DIR, "src"))

SMOKE_OUT = "/content/drive/MyDrive/exp2_geneval_smoke"

!python scripts/run_exp2_sdxl_geneval.py \
    --test-run \
    --device cuda \
    --steps 50 \
    --cfg 5.5 \
    --out {SMOKE_OUT}

## Cell 4 — Verify smoke test output

In [ ]:
import json
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt

manifest_path = Path(SMOKE_OUT) / "manifest.jsonl"
rows = [json.loads(l) for l in manifest_path.read_text().splitlines()]
print(f"{len(rows)} rows in manifest")

for row in rows:
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    fig.suptitle(row["prompt"], fontsize=10)
    for ax, key, label in zip(axes,
        ["standard_path", "resampling_path", "zigzag_path"],
        ["Standard", "Resampling", "Z-Sampling"]):
        ax.imshow(Image.open(row[key]))
        ax.set_title(label)
        ax.axis("off")
    plt.tight_layout()
    plt.show()

## Cell 5 — Full run (all GenEval prompts)

Saves images directly to Google Drive. If the session disconnects, re-run this cell — already-generated images will be skipped automatically.

**Estimated time:** ~12–20 h on T4 (free), ~4–6 h on A100 (Colab Pro).

In [ ]:
FULL_OUT = "/content/drive/MyDrive/exp2_geneval_full"

!python scripts/run_exp2_sdxl_geneval.py \
    --device cuda \
    --steps 50 \
    --cfg 5.5 \
    --out {FULL_OUT}

## Cell 6 — Check progress after a session resumes

In [ ]:
from pathlib import Path

image_dir = Path(FULL_OUT) / "images"
if image_dir.exists():
    std = list(image_dir.glob("*_standard.png"))
    res = list(image_dir.glob("*_resampling.png"))
    zig = list(image_dir.glob("*_zigzag.png"))
    print(f"Standard:   {len(std)} images")
    print(f"Resampling: {len(res)} images")
    print(f"Z-Sampling: {len(zig)} images")
else:
    print("No output directory yet — full run has not started.")